# 02 - Train MLP Models

Train the 20 MLP regressors from the processed artefacts, clearing only `models/MLP/` before the rerun so RF and XGBoost tables are left untouched.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys

project_root = '/content/drive/MyDrive/Dissertation_ML_Project_V1'
if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)
print(f'Project root set to: {project_root}')
print('Contents:', os.listdir('.'))


In [ ]:
import os
import shutil

from src.train_mlp import train_all_mlp

if os.path.exists('models/MLP'):
    shutil.rmtree('models/MLP')
    print('Removed models/MLP for a clean MLP rerun.')
else:
    print('models/MLP does not exist yet; starting from a clean state.')

train_all_mlp('data/processed', 'models/MLP', 'tables')


In [ ]:
import json
import pandas as pd

with open("models/MLP/mlp_results.json") as f:
    results = json.load(f)

rows = []
for target, r in results.items():
    if "error" in r:
        rows.append({"target": target, "status": "ERROR"})
        continue
    rows.append({
        "target": target,
        "r2_cv": r.get("r2_cv"),
        "r2_test": r.get("r2_test"),
        "r2_train": r.get("r2_train"),
        "rmse": r.get("rmse"),
        "mae": r.get("mae"),
        "MU": r.get("max_underestimate"),
        "refined": r.get("refined", False),
        "status": r.get("status"),
        "time_min": r.get("train_time_seconds", 0) / 60,
    })

df = pd.DataFrame(rows)
print(f"Total models: {len(df)}")
print(f"R² >= 0.96: {(df['r2_test'] >= 0.96).sum()}/20")
print(f"Refined: {df['refined'].sum()}/20")
df